## Transform Results Data (Bronze → Silver)
Steps:
1. Read results.csv from Bronze
2. Add ingestion timestamp
3. Select and rename required columns
4. Register table in catalog
5. Write Delta files to ADLS

In [0]:
%run "/Workspace/f1-project/00-Configurations"

In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
results_df = spark.read.option("header", True).option("inferSchema", True).option("nullValue", "\\N").csv(f"{bronze_path}/results.csv")

In [0]:
results_df.show()

+--------+------+--------+-------------+------+----+--------+------------+-------------+------+----+-----------+------------+----------+----+--------------+---------------+--------+
|resultId|raceId|driverId|constructorId|number|grid|position|positionText|positionOrder|points|laps|       time|milliseconds|fastestLap|rank|fastestLapTime|fastestLapSpeed|statusId|
+--------+------+--------+-------------+------+----+--------+------------+-------------+------+----+-----------+------------+----------+----+--------------+---------------+--------+
|       1|    18|       1|            1|    22|   1|       1|           1|            1|  10.0|  58|1:34:50.616|     5690616|        39|   2|      1:27.452|          218.3|       1|
|       2|    18|       2|            2|     3|   5|       2|           2|            2|   8.0|  58|     +5.478|     5696094|        41|   3|      1:27.739|        217.586|       1|
|       3|    18|       3|            3|     7|   7|       3|           3|            3|  

In [0]:
results_df.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: integer (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: integer (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: integer (nullable = true)
 |-- fastestLap: integer (nullable = true)
 |-- rank: integer (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: double (nullable = true)
 |-- statusId: integer (nullable = true)



In [0]:
results_with_ingestion_df = results_df.withColumn("ingestion_date", current_timestamp())

In [0]:
results_selected_df = results_with_ingestion_df.select(
    col("resultId").alias("result_id"),
    col("raceId").alias("race_id"),
    col("driverId").alias("driver_id"),
    col("constructorId").alias("constructor_id"),
    col("grid"),
    col("position"),
    col("points"),
    col("laps"),
    col("time"),
    col("fastestLap").alias("fastest_lap"),
    col("ingestion_date")
)

In [0]:
results_selected_df.show()

+---------+-------+---------+--------------+----+--------+------+----+-----------+-----------+--------------------+
|result_id|race_id|driver_id|constructor_id|grid|position|points|laps|       time|fastest_lap|      ingestion_date|
+---------+-------+---------+--------------+----+--------+------+----+-----------+-----------+--------------------+
|        1|     18|        1|             1|   1|       1|  10.0|  58|1:34:50.616|         39|2026-03-14 11:08:...|
|        2|     18|        2|             2|   5|       2|   8.0|  58|     +5.478|         41|2026-03-14 11:08:...|
|        3|     18|        3|             3|   7|       3|   6.0|  58|     +8.163|         41|2026-03-14 11:08:...|
|        4|     18|        4|             4|  11|       4|   5.0|  58|    +17.181|         58|2026-03-14 11:08:...|
|        5|     18|        5|             1|   3|       5|   4.0|  58|    +18.014|         43|2026-03-14 11:08:...|
|        6|     18|        6|             3|  13|       6|   3.0|  57|  

In [0]:
results_final_df = results_selected_df

In [0]:
results_final_df = results_final_df.dropDuplicates(["result_id"])

In [0]:
results_final_df.count()

27238

In [0]:
%sql
USE CATALOG f1_catalog

In [0]:
%sql
USE SCHEMA f1_silver

In [0]:
%sql
-- DROP TABLE results_1

In [0]:
results_final_df.write.mode("overwrite").format("delta").partitionBy("race_id").saveAsTable('results_1')

In [0]:
%sql
SELECT COUNT(*) FROM f1_silver.results_1

count(1)
27238


In [0]:
%sql
SELECT * FROM f1_silver.results_1 LIMIT 10

result_id,race_id,driver_id,constructor_id,grid,position,points,laps,time,fastest_lap,ingestion_date
7565,1,17,9,8,12,0.0,57,null,38,2026-03-14T11:10:29.502938Z
7573,1,1,1,18,null,0.0,58,+2.914,39,2026-03-14T11:10:29.502938Z
7557,1,10,7,19,4,5.0,58,+4.435,53,2026-03-14T11:10:29.502938Z
7563,1,2,2,9,10,0.0,58,+7.085,48,2026-03-14T11:10:29.502938Z
7569,1,13,6,6,null,0.0,45,null,30,2026-03-14T11:10:29.502938Z
7568,1,8,6,7,15,0.0,55,null,35,2026-03-14T11:10:29.502938Z
7561,1,7,5,17,8,1.0,58,+6.298,50,2026-03-14T11:10:29.502938Z
7558,1,4,4,10,5,4.0,58,+4.879,53,2026-03-14T11:10:29.502938Z
7566,1,20,9,3,13,0.0,56,null,8,2026-03-14T11:10:29.502938Z
7559,1,3,3,5,6,3.0,58,+5.722,48,2026-03-14T11:10:29.502938Z


Write in ADLS

In [0]:
results_final_df.write.mode("overwrite").option("overwriteSchema","true").format("delta").partitionBy("race_id").save(f"{silver_path}/results")